# ゼロから作る Deep Learning ❸ 輪読会
## 第2ステージ「自然なコードで表現する」 ― ステップ 11 〜 17

### これまでの内容
第 1 ステージ (ステップ 1〜10) で、自動微分フレームワーク DeZero のコアを実装した。  

- `Variable` (値 `data` と微分 `grad` を持つ「箱」)

- `Function` (`creator` でつながりを記録し、`forward`/`backward` を持つ関数)

- `y.backward()` で一直線の計算グラフなら自動で微分が求まる

ただし、まだ「入力1つ・出力1つ」「一直線のグラフ」にしか対応していない。  

### このステージの目標
このステージでは、DeZero を「自然なコードで書ける」実用的なフレームワークへと拡張することを目指す。  

1. 複数入力・複数出力の関数 (足し算など) を扱えるようにする (ステップ 11〜13)

2. 同じ変数の再利用や分岐・合流がある複雑なグラフでも正しく微分 (ステップ 14〜16)

3. メモリ管理を改善し、実用に耐えるようにする (ステップ 17)

このステージを終えると、「$z = x^2 + y^2$」のような計算が

```python
z = add(square(x), square(y))
z.backward()   # x.grad, y.grad が自動で求まる
```

と自然に書けるようになる。  

---

| ステップ | テーマ | ゴール |
|---|---|---|
| 11 | 可変長引数 (順伝播編) | 入出力をリストにして複数変数に対応 |
| 12 | 可変長引数 (改善編) | `*inputs` で自然に書けるように改善 |
| 13 | 可変長引数 (逆伝播編) | `Add` の逆伝播を実装、複数出力の逆伝播に対応 |
| 14 | 同じ変数を繰り返し使う | 微分の「加算」と `cleargrad` |
| 15 | 複雑な計算グラフ (理論編) | 逆伝播の正しい順番 ＝「世代」の考え方 |
| 16 | 複雑な計算グラフ (実装編) | `generation` で世代順に逆伝播 |
| 17 | メモリ管理と循環参照 | `weakref` で循環参照を解消 |


## ライブラリ読み込み


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 第 1 ステージ実装したコード

このノートブックは単独で動くように、第 1 ステージ (ステップ 10) の到達点コードを再掲する。  
ここから第 2 ステージの拡張を少しずつ加えていく。  

現状のコードのポイント：  

- `Variable` は `data` / `grad` / `creator` を持ち、`backward()` はループ版。  

- `Function` は入力 `input`・出力 `output` を 1 つずつ覚える (＝ まだ複数入力に非対応)。  

In [ ]:
# ===== 第 1 ステージ (ステップ 10) 時点のコード =====
def as_array(x):
    # 0 次元 ndarray 対策：スカラなら ndarray に変換
    if np.isscalar(x):
        return np.array(x)
    return x

class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output       # 入力・出力は 1 つずつ (今後ここを複数対応にする)
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))
        output.set_creator(self)
        self.input = input
        self.output = output
        return output
    def forward(self, x):
        raise NotImplementedError()
    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        return 2 * self.input.data * gy

print("第 1 ステージの到達点コードを読み込んだ")


---
# ステップ 11：可変長の引数

これまで DeZero の関数は「入力 1 つ・出力 1 つ」だけだった (`square(x)`, `exp(x)`)。しかし、足し算のように入力が複数ある関数もある。  

$$ y = x_0 + x_1 $$

また、多次元配列を分割する関数のように、出力が複数の場合もある。  
こうした可変長 (引数や戻り値の数が 1, 2, 3, … と変化する) に対応できるよう、`Function` を拡張する。  

## 11.1 `Function` クラスの修正

アイデアはシンプルで、入力も出力もリストにまとめることにする。  
これまで 1 つの変数を扱っていたところを、「変数のリスト」を扱うように変えるだけ。  

- 入力：`inputs` (`Variable` のリスト)

- 各データを取り出す：`xs = [x.data for x in inputs]` (リスト内包表記)

- 計算：`ys = self.forward(xs)`

- 出力：`outputs` (`Variable` のリスト)

> リスト内包表記 `[x.data for x in inputs]` は、「`inputs` の各要素 `x` について `x.data` を取り出し、新しいリストを作る」という書き方。  
> `for` ループでリストを作るのを 1 行で書ける。  


In [ ]:
class Function:
    def __call__(self, inputs):          # 引数はリスト
        xs = [x.data for x in inputs]    # 各 Variable からデータを取り出してリスト化
        ys = self.forward(xs)            # 計算 (結果もリストで返す想定)
        outputs = [Variable(as_array(y)) for y in ys]  # 各結果を Variable に包む
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs             # 入力 (複数) を保存
        self.outputs = outputs           # 出力 (複数) を保存
        return outputs

    def forward(self, xs):
        raise NotImplementedError()
    def backward(self, gys):
        raise NotImplementedError()

print("可変長 (リスト版) Function を定義した")


## 11.2 `Add` クラスの実装

複数入力の具体例として足し算 `Add` を実装する。  
引数はリスト `xs` で渡されるので、`x0, x1 = xs` で取り出す。  
戻り値はタプル `(y,)` で返す (複数出力に備えて、出力もまとめる形にするため)。  

In [ ]:
class Add(Function):
    def forward(self, xs):
        x0, x1 = xs        # リストから 2 つの入力を取り出す
        y = x0 + x1
        return (y,)        # 出力はタプルで返す (要素 1 つでも (y,) の形)

# 使ってみる
xs = [Variable(np.array(2)), Variable(np.array(3))]  # 入力をリストで用意
f = Add()
ys = f(xs)             # ys はタプル
y = ys[0]              # 最初の要素を取り出す
print("2 + 3 =", y.data)


正しく `2 + 3 = 5` が計算できた。  
複数入力 (リスト)・複数出力 (タプル) に対応できた。  

ただ、使う側からすると少し不便。  
入力をわざわざリストにまとめ、出力はタプルから取り出す必要がある。  
次のステップで、この使い勝手を改善する。  

> ステップ 11 のまとめ
>
> - `Function` の入出力をリスト/タプルにして、複数入力・複数出力に対応 (順伝播のみ)。  
>
> - `Add` を実装。ただし `f([x0, x1])[0]` のように書くのは不自然 → 次のステップで改善する。  
>

---
# ステップ 12：可変長の引数

ステップ 11 の実装を、2 つの視点で改善する。  

1. 使う人の視点：`f(x0, x1)` と自然に書け、結果も変数を直接受け取れるようにしたい。  

2. 実装する人の視点：`forward(self, x0, x1)` と引数を直接受け取り、`return y` と直接返せるようにしたい。  

## 12.1 使いやすく

`Function.__call__` を 2 点修正する。  

修正①：引数に `*` を付ける (可変長引数)  

`def __call__(self, *inputs)` とすると、`f(x0, x1)` のように任意個の引数を渡せ、それらが `inputs` というタプルにまとまる。  
呼ぶ側はリストを作らなくてよくなる。  

```python
def f(*x):
    print(x)
f(1, 2, 3)   # -> (1, 2, 3) ← まとめて受け取れる
```

修正②：出力が 1 つならリストでなく中身を返す

`return outputs if len(outputs) > 1 else outputs[0]` とすると、出力が 1 つのときは変数を直接返す。  
使う側がタプルから取り出す必要がなくなる。  


In [ ]:
class Function:
    def __call__(self, *inputs):         # 修正①： * を付けて可変長引数に
        xs = [x.data for x in inputs]
        ys = self.forward(xs)
        outputs = [Variable(as_array(y)) for y in ys]
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        self.outputs = outputs
        # 修正②：出力が 1 つなら、その 1 つを直接返す
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, xs):
        raise NotImplementedError()
    def backward(self, gys):
        raise NotImplementedError()

print("使いやすくなった Function (*inputs 版) を定義した")


## 12.2 実装しやすく

`Add` を実装する人にとっても、`forward(self, xs)` の中で `x0, x1 = xs` と分解するのは面倒くさい。  
`forward(self, x0, x1)` と直接受け取り、`return y` と直接返せるようにする。  

そのために `__call__` をさらに 2 点修正する。  

- `ys = self.forward(*xs)`：  
  `*` を付けてアンパック (展開) して渡す。  
  `xs = [x0, x1]` なら `self.forward(x0, x1)` と呼ばれる。  

- `if not isinstance(ys, tuple): ys = (ys,)`：  
  `forward` が単一の値を返しても、内部でタプルに揃える。


In [ ]:
class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)            # * でアンパックして渡す (forward(x0, x1) になる)
        if not isinstance(ys, tuple):     # 単一の戻り値ならタプルに揃える
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        self.outputs = outputs
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, x):
        raise NotImplementedError()
    def backward(self, gy):
        raise NotImplementedError()


# Add を「直接受け取り・直接返す」形で実装できる
class Add(Function):
    def forward(self, x0, x1):   # 引数を直接受け取る
        y = x0 + x1
        return y                 # 結果を直接返す (タプルにしなくてよい)

print("forward も自然に書ける Function / Add を定義しました")


## 12.3 `add` 関数の実装

最後に、`Add` を Python の関数として使えるようにラッパー `add` を用意する。  
これで `add(x0, x1)` と書ける。  


In [ ]:
def add(x0, x1):
    return Add()(x0, x1)

# 自然に書けるようになった
x0 = Variable(np.array(2))
x1 = Variable(np.array(3))
y = add(x0, x1)
print("add(2, 3) =", y.data)


ステップ11では `f([x0, x1])[0]` のように書いていたものが、`add(x0, x1)` とすっきり書けるようになった。  

> ステップ 12 のまとめ
>
> - `*inputs` による可変長引数と「出力 1 つなら直接返す」で、使いやすくした。
>
> - `forward(*xs)` のアンパックとタプル揃えで、簡単に書けるようにした (`forward(self, x0, x1)` と `return y` が書ける)。
>
> - まだ順伝播の改良をしただけ。次は逆伝播を可変長対応にする。


---
# ステップ 13：可変長の引数

順伝播が可変長に対応したので、次は逆伝播を改良する。  

## 13.1 足し算の逆伝播

足し算 $y = x_0 + x_1$ の逆伝播を考える。  
順伝播は「入力 2 つ・出力 1 つ」だが、逆伝播は逆に「入力 1 つ・出力 2 つ」になる。  

まず微分を求める。$y = x_0 + x_1$ を $x_0$ で微分すると：

$$ \frac{\partial y}{\partial x_0} = 1, \qquad \frac{\partial y}{\partial x_1} = 1 $$

> 偏微分 (へんびぶん) とは：
> 複数の入力変数を持つ関数で、1 つの変数だけに注目し、他は定数とみなして微分すること。  
> 記号 $\partial$ (ラウンドディー) を使う。  
> $\frac{\partial y}{\partial x_0}$ は「$x_1$ を定数とみなして $x_0$ で微分」の意味。  
> 足し算では $x_1$ を定数とみなすと、$y = x_0 + (\text{定数})$ なので微分は $1$ ということになる。  

チェインルールより、上流から伝わる微分 `gy` に、この局所的な微分 $1$ を掛けたものが各入力へ流れる。  
$1$ を掛けるだけなので、足し算の逆伝播は「上流の微分をそのまま 2 つに流す」だけ。  

$$ \frac{\partial y}{\partial x_0}\text{側へ}：1 \times gy = gy, \qquad \frac{\partial y}{\partial x_1}\text{側へ}：1 \times gy = gy $$

よって `Add.backward` は `return gy, gy` (2 つの入力に同じ `gy` を返す) となる。  


In [ ]:
class Add(Function):
    def forward(self, x0, x1):
        y = x0 + x1
        return y
    def backward(self, gy):
        # 足し算の逆伝播：上流の微分 gy を、そのまま 2 つの入力へ流す
        return gy, gy    # (x0 側の微分, x1 側の微分)

print("逆伝播つき Add を定義した")


## 13.2 `Variable` クラスの修正

`Add.backward` は「入力 1 つ `gy`、出力 2 つ `(gy, gy)`」である。  
この複数出力を受け止められるよう、`Variable.backward` を修正する。  

これまでは入出力が 1 つずつ前提だが、これを複数対応にする。  
4 つの変更点がある。  

1. `gys = [output.grad for output in f.outputs]`：出力側の微分をリストにまとめる。

2. `gxs = f.backward(*gys)`：`*` でアンパックして渡す。

3. `if not isinstance(gxs, tuple): gxs = (gxs,)`：戻り値が単一ならタプルに揃える。

4. `for x, gx in zip(f.inputs, gxs)`：入力と微分をペアで対応付けして設定。

> `zip` とは：複数のリストを「同じ位置の要素どうし」でペアにする関数。  
> `zip([a,b], [1,2])` は `(a,1), (b,2)` を順に返す。  
> ここでは「$i$ 番目の入力 `f.inputs[i]` の微分は `gxs[i]`」という対応を作っている。  


In [ ]:
class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            gys = [output.grad for output in f.outputs]  # ①出力側の微分をリスト化
            gxs = f.backward(*gys)                       # ②アンパックして backward
            if not isinstance(gxs, tuple):               # ③単一ならタプルに
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):             # ④入力と微分をペアで対応
                x.grad = gx
                if x.creator is not None:
                    funcs.append(x.creator)

print("複数入出力に対応した Variable を定義した")


## 13.3 `Square` クラスの修正

`Function` のインスタンス変数が `input` (単数) から `inputs` (複数) に変わったので、`Square.backward` も `self.inputs[0].data` と書き換える (入力リストの0番目を取る)。  


In [ ]:
class Square(Function):
    def forward(self, x):
        y = x ** 2
        return y
    def backward(self, gy):
        x = self.inputs[0].data   # inputs (複数) の 0 番目
        gx = 2 * x * gy
        return gx

def square(x):
    return Square()(x)

print("複数対応版の Square / square を定義した")


## 動作の確認

いよいよ、複数入力の計算の微分を求める。  

$$ z = x^2 + y^2 $$

を $x=2.0,\ y=3.0$ で計算し、微分を求める。  

手計算では：

$$ \frac{\partial z}{\partial x} = 2x = 4, \qquad \frac{\partial z}{\partial y} = 2y = 6 $$

DeZero では `z = add(square(x), square(y))` と自然に書けて、`z.backward()` で自動微分できる。  


In [ ]:
x = Variable(np.array(2.0))
y = Variable(np.array(3.0))
z = add(square(x), square(y))   # z = x^2 + y^2

z.backward()
print("z      =", z.data,    "  (期待値 2^2 + 3^2 = 13)")
print("x.grad =", x.grad,    "  (期待値 2x = 4)")
print("y.grad =", y.grad,    "  (期待値 2y = 6)")


期待どおり `z=13.0`, `x.grad=4.0`, `y.grad=6.0` が得られた。  
複数入出力に対応した自動微分が完成した。  

> ステップ 13 のまとめ
>
> - 足し算の逆伝播は「上流の微分をそのまま 2 方向へ流す」(偏微分がどちらも 1 のため)。  
>
> - `Variable.backward` を `gys`/`zip`/アンパックで複数入出力対応に。  
>
> - `z = add(square(x), square(y))` のような計算が自然に書けて自動微分できる。  
>
> - ただし、まだ同じ変数を繰り返し使うと誤りが生じるという問題が潜んでいる (次ステップ)。  


---
# ステップ 14：同じ変数を繰り返し使う

前ステップの DeZero には、まだ問題が潜んでいる。  
それは、同じ変数を繰り返し使うときに現れる。  

たとえば：  

$$ y = x + x $$

これは `y = add(x, x)` と書ける。  
数式では $y = 2x$ なので、微分は $\frac{\partial y}{\partial x} = 2$ のはず。  
しかし、現状の DeZero で計算すると…  


In [ ]:
# このセルは「バグを再現する」ためのもの。前半の (ステップ 13 の) Variable を使う
x = Variable(np.array(3.0))
y = add(x, x)
print("y      =", y.data, "  (3 + 3 = 6、これは正しい)")
y.backward()
print("x.grad =", x.grad, "  (期待される値は 2 なのに…？)")


`x.grad` が `1.0` になっていた (正しくは `2.0`)。  
順伝播は正しいのに、逆伝播だけ間違うのです。  

## 14.1 問題の原因

原因は、`Variable.backward` の次の 1 行にある。  

```python
for x, gx in zip(f.inputs, gxs):
    x.grad = gx    # ← ここが問題！
```

`add(x, x)` では、同じ変数 `x` が 2 回入力に使われている。  
逆伝播では `x` に 2 つの微分 (それぞれ `1`) が伝わってくるはず。  
正しくは $1 + 1 = 2$ と足し合わせるべきなのに、`x.grad = gx` では後から来た微分で上書きされてしまい、最後の `1` だけが残る。  

$$ \text{正しい}: \frac{\partial y}{\partial x} = \underbrace{1}_{\text{1つ目のx}} + \underbrace{1}_{\text{2つ目のx}} = 2 $$


## 14.2 解決策

解決策は簡単である。  
「初めての微分なら代入、2 回目以降は加算」に変える。  

```python
if x.grad is None:
    x.grad = gx              # 初めてなら代入
else:
    x.grad = x.grad + gx     # 2 回目以降は加算
```

> なぜ `x.grad += gx` ではなく `x.grad = x.grad + gx`？  
> `+=` (累算代入) は同じメモリを書き換える「インプレース演算」で、別の変数が同じ配列を参照している場合に予期せぬ副作用を起こすことがある。  
> ここでは新しい配列を作る `x.grad = x.grad + gx` を使う (詳細は書籍の付録 A)。  


In [ ]:
class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def cleargrad(self):
        # 微分をリセットするメソッド (14.3 で説明)
        self.grad = None

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            gys = [output.grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx           # 初めては代入
                else:
                    x.grad = x.grad + gx  # 2 回目以降は加算
                if x.creator is not None:
                    funcs.append(x.creator)

print("微分を加算する Variable を定義した")


In [ ]:
# 修正後：y = x + x の微分は正しく 2 になる
x = Variable(np.array(3.0))
y = add(x, x)
y.backward()
print("y = x + x  -> x.grad =", x.grad, "  (正解は 2)")

# y = x + x + x なら微分は 3
x = Variable(np.array(3.0))
y = add(add(x, x), x)
y.backward()
print("y = x+x+x  -> x.grad =", x.grad, "  (正解は 3)")


## 14.3 微分をリセットする

微分を「加算」するようにしたことで、新たな注意点が生じる。  
同じ変数を使い回して別の計算をすると、前の計算の微分が残っていて加算されてしまう。  


In [ ]:
# 同じ x を使い回して2回計算すると…
x = Variable(np.array(3.0))

# 1 回目
y = add(x, x)
y.backward()
print("1 回目 x.grad =", x.grad, "  (正解は 2)")

# 2 回目 (cleargrad しないと前の grad に加算されてしまう)
y = add(add(x, x), x)
y.backward()
print("2 回目 x.grad =", x.grad, "  (誤り：前の 2 が残って 2+3=5 に！)")

2回目が `5.0` (正しくは `3.0`) になった。これは前の微分 `2` が残ったまま加算されたためである。  

これを防ぐのが `cleargrad` メソッド (中身は `self.grad = None`) である。新しい計算の逆伝播の前に呼べば、微分がリセットされる。  


In [ ]:
x = Variable(np.array(3.0))

# 1 回目
y = add(x, x)
y.backward()
print("1 回目 x.grad =", x.grad)

# 2 回目 (今度は cleargrad でリセットしてから)
x.cleargrad()          # 微分をリセット
y = add(add(x, x), x)
y.backward()
print("2 回目 x.grad =", x.grad, "  (正解は 3)")

> ステップ 14 のまとめ
>
> - 同じ変数を繰り返し使うと、微分は加算する必要がある (上書きだと誤る)。  
>
> - `cleargrad` で微分をリセットし、変数を使い回した別計算に対応。  
>
> - しかし、枝分かれ・合流のある複雑なグラフでは、逆伝播の「順番」の問題が残る (次ステップ)。  


---
# ステップ 15：複雑な計算グラフ

これまで扱ったのは一直線の計算グラフだった。しかし、DeZero は今や分岐して合流するといったグラフも作れる。  
ところが、そういうグラフでは逆伝播が誤った順番で行われ、間違った微分の結果を出力してしまう。  
このステップでは、その原因と解決の理論を学ぶ (実装は次ステップ)。  

## 15.1 問題となるグラフ

次のような、途中で分岐して合流するグラフを考える ($a$ が 2 方向に分岐し、後で合流)。

```
        ┌─ B ─ b ─┐
x ─ A ─ a         D ─ y
        └─ C ─ c ─┘
```

このグラフの逆伝播で注目すべきは変数 $a$ である。  
$a$ は $B$ と $C$ の両方に使われているので、$a$ の微分 $\frac{\partial y}{\partial a}$ を求めるには、$B$ 側と $C$ 側の両方から伝わる微分を合流 (加算) する必要がある。  

つまり、$a$ から $x$ (関数 $A$) への逆伝播は、$B$ と $C$ の逆伝播が両方終わってから行わなければいけない。
関数の順で言えば：  

$$ D \to (B\ \text{と}\ C) \to A $$

という順序を守る必要がある ($B$ と $C$ の間の順は任意)。


## 15.2 現状の DeZero の問題

現状の `backward` は、処理すべき関数を `funcs` リストの末尾に追加 (`append`) し、末尾から取り出す (`pop`) だけだった。  
これは「後入れ先出し (スタック)」という処理になる。  

この方式だと、上のグラフでは処理の順番が $D \to C \to A \to B \to A$ となってしまう。  
問題は 2 つ：  

- $C$ の直後に $A$ が処理されてしまう (本来は $B$ も終えてから $A$ にすべき)。  
- 関数 $A$ の逆伝播が 2 回呼ばれてしまう。  

一直線のグラフでは `funcs` に常に 1 個しか入らないので問題にならなかったが、計算が分岐するとこの処理では破綻する。  


## 15.3 解決のアイデア

正しい順番にするには、関数に優先度を付けて、優先度の高い (より出力に近い) 関数から処理すればよいはず。  

その優先度を、難しいグラフ解析なしで得る方法がある。  
それは順伝播のときに「世代（generation）」を記録すること。  

順伝播では「どの関数がどの変数を生んだか」という親子関係に注目する。  
そこで、入力に近い関数から順に第 0 世代、第 1 世代、… と世代を振る。  

```
   世代0   世代1        世代2
x ─ A ─ a ─ B ─ b ─┐
           C ─ c ─ D ─ y
```

逆伝播では、世代の大きい関数から先に処理すれば、「子 (大きい世代) を親 (小さい世代) より先に」処理でき、合流を正しく待てる。  
上の例なら、$A$ (世代 0) より $B, C$ (世代 1) を先に処理できます。  

> 直感的には「世代 = その関数が順伝播で何番目に登場したかの深さ」。  
>
> 深いところ (出力寄り) ほど世代が大きく、逆伝播では深いところから戻る。  

次のステップで、この「世代」を実装する。  

> ステップ 15 のまとめ
>
> - 枝分かれ・合流グラフでは、合流点の微分を正しく加算するため 逆伝播の順番が重要。  
>
> - 現状のスタック方式 (append/pop) では誤った順番で計算が処理される。  
>
> - 順伝播で 世代 (generation) を記録し、逆伝播で世代の大きい順に処理すれば解決。  

---
# ステップ 16：複雑な計算グラフ

ステップ 15 の理論を実装する。  
やることは 2 つある。  

1. 順伝播で世代を設定する (`Variable` と `Function` に `generation` を追加)。
2. 逆伝播で世代の大きい順に関数を取り出す。

## 16.1 世代の追加

`Variable`：`generation` を `0` で初期化。`set_creator` されたとき (＝関数に生み出されたとき)、親の関数より 1 つ大きい世代にする。  

`Function`：入力変数たちの世代の最大値を自分の世代にする (複数入力なら一番大きいものに合わせる)。  


In [ ]:
class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0                 # 世代 (初期値 0)

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1  # 親関数より 1 つ大きい世代に

    def cleargrad(self):
        self.grad = None

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        # 世代順に関数を取り出す仕組み
        funcs = []
        seen_set = set()   # 同じ関数を重複追加しないための集合

        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)  # 世代の小さい順にソート

        add_func(self.creator)

        while funcs:
            f = funcs.pop()   # 末尾 ＝ 世代が最大の関数が取り出される
            gys = [output.grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx
                if x.creator is not None:
                    add_func(x.creator)   # append ではなく add_func 経由

print("世代つき Variable を定義した")


逆伝播の仕組みのポイントを補足する。  

- `add_func` の中で `funcs.sort(key=lambda x: x.generation)` により、`funcs` を世代の小さい順に並べる。  

- `funcs.pop()` はリストの末尾 (＝ 世代最大) を取り出すので、常に「一番出力に近い関数」から処理される。  

- `seen_set` は、同じ関数を 2 回リストに入れない (＝ 逆伝播を 2 回呼ばない) ためのもの。  
  ステップ 15 で見た「$A$ が 2 回呼ばれる」問題を防ぐ。  

> `sort(key=lambda x: x.generation)` は「各要素 `x` の `x.generation` の値を基準に並べ替え」の意味。  
>
> `lambda` は使い捨ての小さな関数を書く記法。  

`Function` にも世代設定を加える。  


In [ ]:
class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        self.generation = max([x.generation for x in inputs])  # 入力の最大世代に合わせる
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        self.outputs = outputs
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, x):
        raise NotImplementedError()
    def backward(self, gy):
        raise NotImplementedError()

# 具体的な関数を新しい Function に合わせて再定義
class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        return 2 * self.inputs[0].data * gy

class Add(Function):
    def forward(self, x0, x1):
        return x0 + x1
    def backward(self, gy):
        return gy, gy

def square(x):
    return Square()(x)
def add(x0, x1):
    return Add()(x0, x1)

print("世代設定つき Function と、具体的関数を定義した")


## 16.4 動作確認

ステップ15で問題になった「分岐・合流」グラフを実際に計算する。  

$$ a = x^2, \qquad y = a^2 + a^2 = 2a^2 = 2x^4 $$

を $x = 2.0$ で計算する。  

手計算では：

$$ \frac{dy}{dx} = 8x^3 = 8 \times 2^3 = 64 $$

`y = add(square(a), square(a))` のように、同じ `a` を分岐させて使いる。  
世代の仕組みが正しく働けば `64.0` になるはず。  


In [ ]:
x = Variable(np.array(2.0))
a = square(x)                     # a = x^2
y = add(square(a), square(a))     # y = a^2 + a^2 = 2a^2 = 2x^4
y.backward()

print("y      =", y.data,  "  (正解は 2 * 2^4 = 32)")
print("x.grad =", x.grad,  "  (正解は 8x^3 = 64)")


`y=32.0`, `x.grad=64.0` と正しく求まった。  
分岐・合流のある複雑なグラフでも、世代順の逆伝播により正しく微分できている。  

これで DeZero は、どれだけ複雑な「つながり」のグラフでも正しく微分できるようになった。  
`Variable` クラスがついに完成した。  

> ステップ 16 のまとめ
>
> - `Variable`/`Function` に `generation` を追加し、順伝播で世代を記録。  
>
> - `backward` で `funcs` を世代順にソートし、世代最大から処理。`seen_set` で重複を防止。  
>
> - 分岐・合流グラフ ($y=2x^4$ など) も正しく微分できるようになった。  


---
# ステップ 17：メモリ管理と循環参照

DeZero は分かりやすさを優先してきたため、メモリ効率には着目してこなかった。  
ニューラルネットワークでは大きなデータを扱うので、メモリ管理は重要。  
このステップでは Python のメモリ管理を学び、DeZero に潜む循環参照を解消する。  

## 17.1〜17.2 Python のメモリ管理

Python (CPython) は主に参照カウントでメモリを管理している。  

- すべてのオブジェクトは「参照カウント (何個から参照されているか)」を持つ。  

- 参照されるとカウント +1、参照が外れると −1。  

- カウントが 0 になった瞬間、即座にメモリから消去される。  

たとえば `a = obj()` でカウント 1、`a = None` でカウント 0 になり消去される。  
この仕組みは高速で、多くの場面でうまく働く。  

## 17.3 循環参照

しかし参照カウントには循環参照 (オブジェクトどうしが輪になって参照し合う) という弱点がある。  

```
a → b → c → a   (c が a を参照して輪になる)
```

この場合、外から `a = b = c = None` としても、3 つは互いに参照し合っているためカウントが 1 のまま 0 になならない。  
誰からもアクセスできないのに、メモリに残り続ける。  

(Python にはこれを回収する GC (世代別ガーベージコレクション) もあるが、回収が遅れてメモリ使用量が増える原因になる。)

### DeZero の循環参照
実は現状の DeZero には循環参照がある。  

```
Variable ──creator──→ Function
Variable ←──outputs── Function   (Function が出力 Variable を参照)
```

`Function` は出力 `Variable` を `outputs` で参照し、その `Variable` は生みの親 `Function` を `creator` で参照する。  
互いに参照し合う繋がりができている。  


## 17.4 `weakref` で循環参照を解消

解決には弱参照 (weak reference) を使う。  
弱参照とは、参照カウントを増やさずに別のオブジェクトを参照する仕組みである。  
Python の `weakref` モジュールで作れる。  

`weakref.ref(obj)` で弱参照を作り、参照先には `b()` のようにカッコを付けてアクセスする。  

DeZero では、`Function` が出力を持つ `self.outputs` を弱参照する。  
これで「$Function \to Variable$」の参照がカウントを増やさなくなり、輪が切れる。  

この変更に伴い、`backward` 内で出力の微分にアクセスする箇所を `output.grad` から `output().grad`（カッコ追加）に変える。  


In [ ]:
import weakref

class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        self.generation = max([x.generation for x in inputs])
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        # 出力を弱参照で持つ (循環参照を断ち切る)
        self.outputs = [weakref.ref(output) for output in outputs]
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, x):
        raise NotImplementedError()
    def backward(self, gy):
        raise NotImplementedError()


class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1

    def cleargrad(self):
        self.grad = None

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = []
        seen_set = set()
        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)
        add_func(self.creator)
        while funcs:
            f = funcs.pop()
            # 弱参照なので output() とカッコを付けてアクセス
            gys = [output().grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx
                if x.creator is not None:
                    add_func(x.creator)


# 具体的関数を最新の Function に合わせて再定義
class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        return 2 * self.inputs[0].data * gy

class Add(Function):
    def forward(self, x0, x1):
        return x0 + x1
    def backward(self, gy):
        return gy, gy

def square(x):
    return Square()(x)
def add(x0, x1):
    return Add()(x0, x1)

print("weakref で循環参照を解消した DeZero (第 2 ステージ完成版) を定義した")


## 17.5 動作確認

弱参照にしても、これまでの計算が正しく動くことを確認します。  
まず微分計算が壊れていないことをチェックします。  


In [ ]:
# 微分計算が正しく動くことを確認（weakref 化しても結果は同じ）
x = Variable(np.array(2.0))
a = square(x)
y = add(square(a), square(a))    # y = 2x^4
y.backward()
print("x.grad =", x.grad, "  (期待値 64)")


次に、循環参照が解消されメモリが正しく解放されることを、大きなデータで繰り返し計算して確認する。  
循環参照が残っていると、ループのたびに古い計算グラフがメモリに残り続ける。  
弱参照化により、各ループの計算グラフは即座に解放されるはず。  


In [ ]:
import gc

# 大きなデータで繰り返し計算し、メモリが増え続けないことを確認
for i in range(10):
    x = Variable(np.random.randn(10000))  # 大きなデータ
    y = square(square(square(x)))          # 複雑な計算
    # 各ループの終わりで x, y が上書きされ、前の計算グラフは参照されなくなる
    # → 循環参照がなければ、参照カウント0で即座にメモリ解放される

# 循環参照が残っていないことの簡易確認（GC が回収する循環参照ゴミの個数）
gc.collect()
garbage = len(gc.garbage)
print(f"GC が抱える回収待ちの循環参照ゴミ：{garbage} 個")
print("（weakref 化により、計算グラフは循環参照を作らずに解放される）")


> ステップ 17 のまとめ
>
> - Python は参照カウントでメモリ管理するが、循環参照は回収されにくい。
>
> - DeZero の `Variable ⇄ Function` は循環参照だった。
>
> - `Function.outputs` を `weakref` にして輪を断ち切り、計算グラフが即座に解放されるように。
>
> - 逆伝播では `output().grad` とカッコ付きでアクセスする。


左（`cleargrad` あり）は $x$ が滑らかに 0 へ収束しているが、右（なし）は勾配が加算され続けて $x$ が暴れ、最小化に失敗している。  

これが PyTorch で学習ループの先頭に `optimizer.zero_grad()` を書く理由である。  
DeZero のステップ 14 で手作りした「勾配は加算される」という仕様が、実務での定番作法に直結していることが体感できた。


---
# 今回のまとめ

ステップ 11〜17 で、DeZero は「単純な計算しかできない」状態から「どんな複雑なつながりの計算でも自然に書けて自動微分できる」実用的なフレームワークへと成長した。

1. ステップ 11〜13（可変長引数）入出力をリスト化し、`*inputs`・アンパック・`zip` で複数入力・複数出力に対応。`add` を実装し `z = add(square(x), square(y))` が書けるように。足し算の逆伝播は偏微分がどちらも1なので「上流の微分をそのまま2方向へ流す」。  

2. ステップ 14（変数の再利用）同じ変数の微分は加算。使い回しには `cleargrad`。  

3. ステップ 15〜16（複雑なグラフ）順伝播で世代（generation）** を記録し、逆伝播で世代の大きい順に処理。  
  分岐・合流グラフも正しく微分。

4. ステップ 17（メモリ管理）`weakref` で `Variable ⇄ Function` の循環参照を解消。  

### 第 1 ステージ → 第 2 ステージ の対応
| 第1ステージ | 第2ステージでの拡張 |
|---|---|
| 入力1つ・出力1つ | `*inputs` で複数入力・複数出力 |
| 一直線のグラフ | 世代管理で分岐・合流グラフ |
| 微分は上書き | 微分は加算（＋ `cleargrad`） |
| メモリ無頓着 | `weakref` で循環参照回避 |

### 次のステップの内容
次のステージでは、`+` や `*` の演算子オーバーロード（`y = a * b + c` と書ける）、メモリをさらに節約する推論モード、そして DeZero を Python パッケージとしてまとめる。  
